# No Naturalness

In [ ]:
import tensorflow as tf
import numpy as np
import os
import cv2
from glob import glob
import matplotlib.pyplot as plt
import pandas as pd

# --- 1. CONFIGURATION ---
DATASET_FOLDER = 'sample_dataset'
METADATA_CSV = os.path.join(DATASET_FOLDER, 'metadata.csv')
MODEL_PATH_DTN = 'models/k3_100epch_wo_custom_loss_model.h5'
MODEL_PATH_YOLO = 'yolov8n_saved_model'
IMG_SIZE = (500, 500)
YOLO_SIZE = (640, 640)
CAR_CLASS_ID = 2 
CLASSES = {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}

# Generate a simulated palette of printable colors (125 colors evenly spaced)
# In the future, you can replace this with an array of specific RGBs from your actual printer
def generate_generic_printer_palette(n_per_channel=5):
    grid_1d = tf.linspace(0.05, 0.95, n_per_channel)  # avoid hard extremes
    r, g, b = tf.meshgrid(grid_1d, grid_1d, grid_1d)
    colors = tf.reshape(tf.stack([r, g, b], axis=-1), [-1, 3])

    # Remove extremely high saturation colors (printer-unfriendly)
    max_channel = tf.reduce_max(colors, axis=1)
    min_channel = tf.reduce_min(colors, axis=1)
    saturation = max_channel - min_channel

    mask = saturation < 0.9  # filter neon-ish colors
    colors = tf.boolean_mask(colors, mask)

    return colors

PRINTABLE_COLORS = generate_generic_printer_palette(5)

try:
    df_meta = pd.read_csv(METADATA_CSV)
except FileNotFoundError:
    print(f"Error: Could not find {METADATA_CSV}")
    exit(1)

# --- 2. MODELS SETUP ---
dtn_model = tf.keras.models.load_model(MODEL_PATH_DTN, compile=False)
yolo_loaded = tf.saved_model.load(MODEL_PATH_YOLO)
yolo_infer = yolo_loaded.signatures['serving_default']

def load_dataset_sample(idx):
    row = df_meta.iloc[idx]
    base_name = str(row['filename']).zfill(5)
    
    def prep_img(folder, ext):
        p = os.path.join(DATASET_FOLDER, folder, f"{base_name}{ext}")
        if not os.path.exists(p):
            p = glob(os.path.join(DATASET_FOLDER, folder, f"{base_name}.*"))[0]
        img = tf.image.decode_image(tf.io.read_file(p), channels=3)
        img = tf.image.convert_image_dtype(img, tf.float32)
        return tf.image.resize(img, IMG_SIZE)

    ref = prep_img('reference', '.png')
    mask = prep_img('masks', '.png')
    overlay = prep_img('overlays', '.png')
    trans = np.array([row['distance'], row['pitch'], row['yaw']], dtype=np.float32)
    
    return ref, mask, overlay, trans

# --- 3. LOGIC FUNCTIONS ---

def calc_transforms_tf(pitch, yaw):
    yaw_diff = ((yaw + 45.0) % 90.0) - 45.0
    cond_front = tf.logical_and(yaw > 145.0, yaw < 215.0)
    
    mid_p = tf.where(cond_front, -pitch, pitch)
    mid_y = tf.where(cond_front, -yaw_diff, 0.0)
    mid_r = tf.where(cond_front, 0.0, -yaw_diff)

    low_cond = pitch < 15.0
    p = tf.where(low_cond, pitch, mid_p)
    y = tf.where(low_cond, 0.0, mid_y)
    r = tf.where(low_cond, -yaw_diff, mid_r)

    top_cond = pitch > 35.0
    p = tf.where(top_cond, -pitch, p)
    y = tf.where(top_cond, -yaw_diff, y)
    r = tf.where(top_cond, 0.0, r)
    return p, y, r

def calculate_nps(texture):
    """Calculates the Non-Printability Score for the texture patch."""
    # Reshape texture to a list of pixels: (N_pixels, 3)
    pixels = tf.reshape(texture, [-1, 3])
    # Expand dims to calculate pairwise differences with palette: (N_pixels, 1, 3) - (1, N_colors, 3)
    pixels_exp = tf.expand_dims(pixels, 1)
    palette_exp = tf.expand_dims(PRINTABLE_COLORS, 0)
    
    # Calculate Euclidean distance between every pixel and every palette color
    distances = tf.sqrt(tf.reduce_sum(tf.square(pixels_exp - palette_exp), axis=-1) + 1e-8)
    
    # Find the distance to the *closest* printable color for each pixel
    min_distances = tf.reduce_min(distances, axis=1)
    
    # The NPS is the sum (or mean) of these minimum distances
    return tf.reduce_mean(min_distances)

#increase tv wight = smoother tex
#increase nps wight = more printable tex (less neon colors)
def calculate_custom_loss(yolo_output, texture_variable, tv_weight=0.05, nps_weight=0.1):
    #{0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 5: 'bus', 7: 'truck'}
    VEHICLE_CLASS_IDS = [2, 7, 5]  # car, truck, bus
    class_scores = [yolo_output[0, 4 + class_id, :] for class_id in VEHICLE_CLASS_IDS]
    stacked_scores = tf.stack(class_scores, axis=0)
    vehicle_scores = tf.reduce_max(stacked_scores, axis=0)
    car_conf = tf.reduce_max(vehicle_scores)

    #to do misclassification rather than detection avoidance:
    TARGET_CLASS_ID = 1
    target_scores = yolo_output[0, 4 + TARGET_CLASS_ID, :]
    target_conf = tf.reduce_max(target_scores)

    # NEGATE because optimizer minimizes
    detection_loss = -(target_conf - car_conf)

    raw_tv = tf.reduce_sum(tf.image.total_variation(texture_variable))
    # 2. Divide by the total number of elements (H * W * C)
    num_elements = tf.cast(tf.size(texture_variable), tf.float32)
    normalized_tv = raw_tv / num_elements

    nps_loss = calculate_nps(texture_variable)
    
    total_loss = detection_loss + (tv_weight * normalized_tv) + (nps_weight * nps_loss)
    return total_loss, detection_loss, normalized_tv, nps_loss

def get_rotation_matrix_tf(pitch, yaw, roll):
    deg2rad = np.pi / 180.0
    p, y, r = pitch * deg2rad, yaw * deg2rad, roll * deg2rad
    rot_pitch = tf.stack([[1.0, 0.0, 0.0], [0.0, tf.cos(p), -tf.sin(p)], [0.0, tf.sin(p),  tf.cos(p)]])
    rot_roll = tf.stack([[tf.cos(r), 0.0, tf.sin(r)], [0.0, 1.0, 0.0], [-tf.sin(r), 0.0, tf.cos(r)]])
    rot_yaw = tf.stack([[tf.cos(y), -tf.sin(y), 0.0], [tf.sin(y),  tf.cos(y), 0.0], [0.0, 0.0, 1.0]])
    return tf.matmul(rot_pitch, tf.matmul(rot_roll, rot_yaw))

def bilinear_sampler_tf(img, x, y):
    H_f, W_f = tf.cast(tf.shape(img)[0], tf.float32), tf.cast(tf.shape(img)[1], tf.float32)
    x, y = tf.clip_by_value(x, 0.0, W_f - 1.001), tf.clip_by_value(y, 0.0, H_f - 1.001)
    x0, y0 = tf.cast(tf.floor(x), tf.int32), tf.cast(tf.floor(y), tf.int32)
    x1, y1 = x0 + 1, y0 + 1
    
    Ia = tf.gather_nd(img, tf.stack([y0, x0], axis=-1))
    Ib = tf.gather_nd(img, tf.stack([y1, x0], axis=-1))
    Ic = tf.gather_nd(img, tf.stack([y0, x1], axis=-1))
    Id = tf.gather_nd(img, tf.stack([y1, x1], axis=-1))
    
    wa = tf.expand_dims((tf.cast(x1, tf.float32) - x) * (tf.cast(y1, tf.float32) - y), -1)
    wb = tf.expand_dims((tf.cast(x1, tf.float32) - x) * (y - tf.cast(y0, tf.float32)), -1)
    wc = tf.expand_dims((x - tf.cast(x0, tf.float32)) * (tf.cast(y1, tf.float32) - y), -1)
    wd = tf.expand_dims((x - tf.cast(x0, tf.float32)) * (y - tf.cast(y0, tf.float32)), -1)
    
    return tf.add_n([wa*Ia, wb*Ib, wc*Ic, wd*Id])

@tf.function
def render_texture_tf(texture, pitch, yaw, roll, distance, uv_scale=200.0, shift_u=0.0, shift_v=0.0):
    out_h, out_w = IMG_SIZE
    f = 500.0
    cx, cy = out_w / 2.0, out_h / 2.0
    
    high_res_tex = tf.image.resize(texture, [256, 256], method='nearest')
    R = get_rotation_matrix_tf(pitch, yaw, roll)
    plane_normal = R[:, 2] 

    pixels_per_meter = 100.0 
    center_dist_units = distance * pixels_per_meter
    plane_point = tf.constant([0.0, 0.0, 0.0]) + (plane_normal * center_dist_units)

    grid_x, grid_y = tf.meshgrid(tf.range(out_w), tf.range(out_h))
    rx, ry, rz = tf.cast(grid_x, tf.float32) - cx, tf.cast(grid_y, tf.float32) - cy, tf.ones_like(tf.cast(grid_x, tf.float32)) * f
    ray_dir = tf.stack([rx, ry, rz], axis=-1)

    camera_pos = tf.constant([0.0, 0.0, -f])
    numerator = tf.tensordot(plane_point - camera_pos, plane_normal, axes=1)
    denominator = tf.where(tf.abs(tf.tensordot(ray_dir, plane_normal, axes=1)) < 1e-5, 1e-5, tf.tensordot(ray_dir, plane_normal, axes=1))
    t = numerator / denominator
    hit_point = camera_pos + (ray_dir * tf.expand_dims(t, -1))

    p_local = tf.reshape(tf.matmul(tf.reshape(hit_point - plane_point, [-1, 3]), R), [out_h, out_w, 3])
    
    # Apply EoT shifts here
    u = (p_local[:, :, 0] / uv_scale) + shift_u
    v = (p_local[:, :, 1] / uv_scale) + shift_v
    
    u, v = tf.math.floormod(u, 1.0), tf.math.floormod(v, 1.0)
    
    tex_h, tex_w = tf.cast(tf.shape(high_res_tex)[0], tf.float32), tf.cast(tf.shape(high_res_tex)[1], tf.float32)
    output = bilinear_sampler_tf(high_res_tex, u * (tex_w - 1.0), v * (tex_h - 1.0))
    
    return tf.where(tf.expand_dims(t > 0.0, -1), output, tf.zeros_like(output))

# --- 4. YOLO VISUALIZATION ---

def draw_yolo_results(image_tf, yolo_output, conf_thresh=0.25):
    img = (image_tf.numpy() * 255).astype(np.uint8).copy()
    output = np.transpose(yolo_output.numpy()[0])
    
    boxes, confs, class_ids = [], [], []
    for row in output:
        scores = row[4:]
        cls_id = np.argmax(scores)
        score = scores[cls_id]
        if score > conf_thresh:
            cx, cy, w, h = row[:4] * (500.0 / 640.0)
            boxes.append([int(cx - w/2), int(cy - h/2), int(w), int(h)])
            confs.append(float(score))
            class_ids.append(cls_id)

    indices = cv2.dnn.NMSBoxes(boxes, confs, conf_thresh, 0.45)
    if len(indices) > 0:
        for i in indices.flatten():
            bx, by, bw, bh = boxes[i]
            cv2.rectangle(img, (bx, by), (bx+bw, by+bh), (0, 255, 0), 2)
            cv2.putText(img, f"{CLASSES.get(class_ids[i], 'obj')}: {confs[i]:.2f}", (bx, by-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
    return img

# --- 5. TRAINING STEP ---

optimizer = tf.keras.optimizers.Adam(learning_rate=0.03)
tf_texture = tf.Variable(np.random.uniform(0, 1, (8, 8, 3)).astype(np.float32))

@tf.function
def train_step(ref_img, mask_img, overlay_img, trans_num_tf, shift_u, shift_v):
    dist, pitch, yaw = trans_num_tf[0], trans_num_tf[1], trans_num_tf[2]
    p, y, r = calc_transforms_tf(pitch, yaw)
    
    with tf.GradientTape() as tape:
        # Pass the EoT shifts to the renderer
        rendered = render_texture_tf(tf_texture, p, y, r, dist, shift_u=shift_u, shift_v=shift_v)
        
        mask_bool = tf.reduce_max(mask_img, axis=-1, keepdims=True) > 0.1
        tex_masked = tf.where(mask_bool, rendered, tf.zeros_like(rendered))
        
        dtn_out = dtn_model([tf.expand_dims(ref_img, 0), tf.expand_dims(tex_masked, 0)], training=False)[0]
        
        overlay_bool = tf.reduce_max(overlay_img, axis=-1, keepdims=True) > 0.05
        adv_img = tf.where(overlay_bool, ref_img, dtn_out)
        
        yolo_in = tf.image.resize(tf.expand_dims(adv_img, 0), YOLO_SIZE)
        yolo_out = yolo_infer(images=yolo_in)['output_0']
        
        # Unpack the new detailed losses
        total_loss, det_loss, tv_loss, nps_loss = calculate_custom_loss(yolo_out, tf_texture)
        
    optimizer.apply_gradients(zip(tape.gradient(total_loss, [tf_texture]), [tf_texture]))
    return total_loss, det_loss, tv_loss, nps_loss, adv_img, yolo_out

# --- 6. RUN VALIDATION ---

ref, mask, overlay, trans = load_dataset_sample(2468)
print("Starting optimization...")

ref_yolo_in = tf.image.resize(tf.expand_dims(ref, 0), YOLO_SIZE)
ref_yolo_out = yolo_infer(images=ref_yolo_in)['output_0']
ref_max_conf = tf.reduce_max(tf.stack([ref_yolo_out[0, 4 + c, :] for c in [2, 7, 5]], axis=0))

trans_num_tf = tf.constant(trans[:3].astype(np.float32))

# Use 0.0 shifts for the baseline visualization
p_init, y_init, r_init = calc_transforms_tf(trans_num_tf[1], trans_num_tf[2])
init_rendered = render_texture_tf(tf_texture, p_init, y_init, r_init, trans_num_tf[0], shift_u=0.0, shift_v=0.0)
init_mask_bool = tf.reduce_max(mask, axis=-1, keepdims=True) > 0.1
init_tex_masked = tf.where(init_mask_bool, init_rendered, tf.zeros_like(init_rendered))
init_dtn = dtn_model([tf.expand_dims(ref, 0), tf.expand_dims(init_tex_masked, 0)], training=False)[0]
init_overlay_bool = tf.reduce_max(overlay, axis=-1, keepdims=True) > 0.05
init_adv = tf.where(init_overlay_bool, ref, init_dtn)
init_yolo = yolo_infer(images=tf.image.resize(tf.expand_dims(init_adv, 0), YOLO_SIZE))['output_0']

# Trackers for the new loss components
history = {'total': [], 'det': [], 'tv': [], 'nps': []}

for step in range(15):
    # EOT: Generate random shifts for this step
    rand_shift_u = tf.random.uniform(shape=[], minval=0.0, maxval=1.0)
    rand_shift_v = tf.random.uniform(shape=[], minval=0.0, maxval=1.0)
    
    t_loss, d_loss, tv_loss, nps_loss, cur_adv, cur_yolo = train_step(
        ref, mask, overlay, trans_num_tf, rand_shift_u, rand_shift_v
    )
    
    history['total'].append(t_loss.numpy())
    history['det'].append(d_loss.numpy())
    history['tv'].append(tv_loss.numpy())
    history['nps'].append(nps_loss.numpy())
    
    print(f"Step {step:3d} | Total: {t_loss.numpy():.4f} | Conf: {d_loss.numpy():.4f} | NPS: {nps_loss.numpy():.4f} | TV: {tv_loss.numpy():.4f}")

# --- 7. FINAL DISPLAY ---
plt.figure(figsize=(18, 10))

plt.subplot(2, 3, 1)
plt.imshow(draw_yolo_results(ref, ref_yolo_out))
plt.title(f"Original Reference (Conf: {ref_max_conf.numpy():.4f})")

plt.subplot(2, 3, 2)
plt.imshow(draw_yolo_results(init_adv, init_yolo))
plt.title("Initial Attack State (No Shift)")

plt.subplot(2, 3, 3)
plt.imshow(draw_yolo_results(cur_adv, cur_yolo))
plt.title(f"Optimized State (Conf: {d_loss.numpy():.4f})")

plt.subplot(2, 3, 4)
# Ensure the texture is clipped to [0,1] for display, simulating physical output
plt.imshow(tf.clip_by_value(tf_texture, 0.0, 1.0).numpy())
plt.title("Optimized Patch")

plt.subplot(2, 3, 5)
plt.plot(history['total'], label='Total Loss', color='black', linewidth=2)
plt.plot(history['det'], label='Detection', linestyle='--')
plt.plot(history['nps'], label='NPS', linestyle='-.')
plt.plot(history['tv'], label='TV', linestyle=':')
plt.title("Loss Components")
plt.xlabel("Optimization Steps")
plt.legend()

plt.tight_layout()
plt.show()